Поставил помимо того что было:

```python
uv pip install xgboost lightgbm optuna
```

In [3]:
%pip install "pandas<3.0.0" lightgbm xgboost catboost optuna scikit-learn==1.8.0


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install lightgbm 


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
sklearn.set_config(transform_output="pandas")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, OrdinalEncoder, TargetEncoder
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, RadiusNeighborsClassifier
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, VotingRegressor
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer


from catboost import CatBoostRegressor, CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb

import optuna
import joblib

In [6]:
train = pd.read_csv("../../data/train.csv")

train.head(10)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
5,6,50,RL,85.0,14115,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,MnPrv,Shed,700,10,2009,WD,Normal,143000
6,7,20,RL,75.0,10084,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,307000
7,8,60,RL,NaN,10382,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,Shed,350,11,2009,WD,Normal,200000
8,9,50,RM,51.0,6120,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2008,WD,Abnorml,129900
9,10,190,RL,50.0,7420,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,1,2008,WD,Normal,118000


In [7]:
X, y = train.drop('SalePrice', axis=1), np.log1p(train['SalePrice'])

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# IMPUTER
cols_drop = [
    "Street",
    "Utilities",
    "PoolQC",
    "PoolArea",
    "MiscFeature",
    "MiscVal",
    "Alley",
]

# Текстовые — нет объекта
text_none = [
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "FireplaceQu",
    "Fence",
    "MasVnrType",
]

# Числовые — нет объекта
num_none = [
    "GarageArea",
    "GarageCars",
    "GarageYrBlt",
    "MasVnrArea",
    "BsmtFullBath",
    "BsmtHalfBath",
    "TotalBsmtSF",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
]

# Группа 2 — мода (1-4 пропуска)
mode_cols = [
    "MSZoning",
    "Functional",
    "Exterior1st",
    "Exterior2nd",
    "Electrical",
    "KitchenQual",
    "SaleType",
]

# Ordinal_data
qual_map = {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
fin_map = {"NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
exp_map = {"NA": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}
grg_map = {"NA": 0, "Unf": 1, "RFn": 2, "Fin": 3}
func_map = {
    "Sal": 0,
    "Sev": 1,
    "Maj2": 2,
    "Maj1": 3,
    "Mod": 4,
    "Min2": 5,
    "Min1": 6,
    "Typ": 7,
}

ordinal_encoding_columns = {
    "ExterQual": qual_map,
    "ExterCond": qual_map,
    "HeatingQC": qual_map,
    "KitchenQual": qual_map,
    "FireplaceQu": qual_map,
    "GarageQual": qual_map,
    "GarageCond": qual_map,
    "BsmtQual": qual_map,
    "BsmtCond": qual_map,
    "BsmtFinType1": fin_map,
    "BsmtFinType2": fin_map,
    "BsmtExposure": exp_map,
    "GarageFinish": grg_map,
    "Functional": func_map,
    "LotShape": {"Reg": 3, "IR1": 2, "IR2": 1, "IR3": 0},
    "LandSlope": {"Gtl": 1, "Mod": 2, "Sev": 3},
    "PavedDrive": {"N": 0, "P": 1, "Y": 2},
    "CentralAir": {"N": 0, "Y": 1},
}

# # LOG TRANSFORM
skewed = [
    "LotArea",
    "LotFrontage",
    "GrLivArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "WoodDeckSF",
    "OpenPorchSF",
    "MasVnrArea",
    "TotalSF",
    "QualSF",
    "QualTotalSF",
    "TotalPorchSF",
]

# # BOOLEAN TRANSFORM

# print(df.dtypes.value_counts())
# print()
# bool_cols = df.select_dtypes(include="bool").columns
# df[bool_cols] = df[bool_cols].astype(int)

# print(df.dtypes.value_counts())

# # ENCODE
# df = pd.get_dummies(df, columns=cat_cols, drop_first=False)
# print(f"Итого колонок после OHE: {df.shape[1]}")

In [9]:
# ДОБАВИТЬ В ПРЕПРОЦЕССОР
class GroupMedianImputer(BaseEstimator, TransformerMixin):
    """
    Заполняет пропуски в целевом столбце медианой по группам.
    """
    def __init__(self, group_col, target_col):
        self.group_col = group_col
        self.target_col = target_col
        self.medians_ = None   # словарь или Series с медианами для каждой группы

    def fit(self, X, y=None):
        # X должен быть DataFrame, содержащим оба столбца
        # Вычисляем медиану для каждой группы только на обучающих данных
        self.medians_ = X.groupby(self.group_col)[self.target_col].median()
        return self

    def transform(self, X):
        X = X.copy()
        # Для каждого значения группы подставляем соответствующую медиану
        # Если в X встречается группа, которой не было в fit, используем общую медиану (опционально)
        for group, median in self.medians_.items():
            mask = (X[self.group_col] == group) & (X[self.target_col].isna())
            X.loc[mask, self.target_col] = median
        
        # Обрабатываем оставшиеся NaN (например, группы, не встречавшиеся в обучении)
        # Можно заполнить глобальной медианой или оставить как есть
        if X[self.target_col].isna().any():
            global_median = X[self.target_col].median()
            X[self.target_col].fillna(global_median, inplace=True)
        return X
    
class Ordinal_mapper(BaseEstimator, TransformerMixin):
    '''
    Обработка по значениям
    '''
    def __init__(self, mapping):
        self.mapping = mapping
        self.unknown_value = -1

    def fit(self, X, y=None):
        for col in self.mapping:
            if col not in X.columns:
                raise ValueError(f"Column '{col}' not found in data")
        return self
    
    def transform(self, X):
        X = X.copy()
        for col, mapping in self.mapping.items():
            X[col] = X[col].map(mapping).fillna(self.unknown_value).astype(int)
        return X
    
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mas_vnr_mode_ = None

    def fit(self, X, y=None):
        # Вычисляем моду на обучающих данных (игнорируя 'NA' как пропуск)
        df = X.copy()
        # Берём только не пропущенные значения (не 'NA')
        non_na = df[df['MasVnrType'] != 'NA']['MasVnrType']
        # Если есть значения, вычисляем моду
        if not non_na.empty:
            self.mas_vnr_mode_ = non_na.mode()[0]
        else:
            self.mas_vnr_mode_ = 'NA'  # fallback
        return self

    def transform(self, X):
        df = X.copy()
        
        # ----- Логика исправления MasVnrType / MasVnrArea -----
        # Кладка есть, но тип пропущен (NA) -> заменяем на моду
        mask1 = (df['MasVnrType'] == 'NA') & (df['MasVnrArea'] > 0)
        df.loc[mask1, 'MasVnrType'] = self.mas_vnr_mode_
        
        # Кладка указана, но площадь 0 -> убираем тип
        mask2 = (df['MasVnrType'] != 'NA') & (df['MasVnrArea'] == 0)
        df.loc[mask2, 'MasVnrType'] = 'NA'
        
        # ----- Далее идут ваши преобразования (создание новых признаков) -----
        # Возраст дома
        df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
        df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
        df["IsRemodeled"] = (df["YearBuilt"] != df["YearRemodAdd"]).astype(int)
        
        # Площади
        df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
        df["TotalPorchSF"] = (
            df["OpenPorchSF"] + df["EnclosedPorch"] + 
            df["ScreenPorch"] + df["WoodDeckSF"]
        )
        
        # Ванные
        df["TotalBath"] = (
            df["FullBath"] + df["BsmtFullBath"] + 
            0.5 * df["HalfBath"] + 0.5 * df["BsmtHalfBath"]
        )
        
        # Флаги
        df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
        df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
        df["HasPorch"] = (df["TotalPorchSF"] > 0).astype(int)
        
        # Interaction
        df["QualSF"] = df["OverallQual"] * df["GrLivArea"]
        df["QualTotalSF"] = df["OverallQual"] * df["TotalSF"]
        
        return df
    
def log_transform(X):
    X = X.copy()
    for col in skewed:
        X[col] = np.log1p(X[col])
    return X

def bool_to_int(X):
    X = X.copy()
    bool_cols = X.select_dtypes(include='bool').columns
    X[bool_cols] = X[bool_cols].astype(int)
    return X

In [10]:
my_imputer = ColumnTransformer(
    transformers = [
        ('drop_features', 'drop', cols_drop),
        ('num_imputer', SimpleImputer(strategy='constant', fill_value=0), num_none),
        ('cat_imputer', SimpleImputer(strategy='constant', fill_value='NA'), text_none),
        ('mode_imputer', SimpleImputer(strategy='most_frequent'), mode_cols),
    ],
    verbose_feature_names_out = False,
    remainder='passthrough'
)

In [11]:
processed_data = my_imputer.fit_transform(X_train)

In [12]:
cat_cols = processed_data.select_dtypes(include="object").columns.tolist()

my_imputer_2 = ColumnTransformer(
    transformers = [
        ('one_hot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols),
    ],
    verbose_feature_names_out = False,
    remainder='passthrough'
)

In [13]:
preprocessor = Pipeline(
    [
        ('imputer', my_imputer),
        ('MedianImputer', GroupMedianImputer(group_col='Neighborhood', target_col='LotFrontage')),
        ('Ordinal_mapper', Ordinal_mapper(ordinal_encoding_columns)),
        ('feature_engineer', FeatureEngineer()),
        ('my_imputer_2', my_imputer_2),
        ('bool_to_int', FunctionTransformer(bool_to_int, validate=False)),
        ('log_transform', FunctionTransformer(log_transform, validate=False)),
        ]
        
)

In [14]:
preprocessor.fit_transform(X_train)

,GarageType_2Types,GarageType_Attchd,GarageType_Basment,GarageType_BuiltIn,GarageType_CarPort,GarageType_Detchd,GarageType_NA,GarageFinish_0,GarageFinish_1,GarageFinish_2,...,RemodAge,IsRemodeled,TotalSF,TotalPorchSF,TotalBath,HasFireplace,HasGarage,HasPorch,QualSF,QualTotalSF
254,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,53,0,7.874359,5.525453,2.0,0,1,1,8.790421,9.483492
1066,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,15,1,7.771067,3.713572,2.5,1,1,1,9.151333,9.562475
638,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,58,1,7.373374,6.200509,1.0,0,0,1,8.289288,8.982310
799,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,57,1,7.824046,5.579730,2.5,1,1,1,9.087155,9.433164
380,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,60,1,7.907652,5.493061,2.0,1,1,1,9.042632,9.516795
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1,0,7.874359,3.135494,2.0,1,1,1,8.972717,9.665801
1130,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,59,1,8.040447,6.165418,3.0,1,1,1,8.977778,9.426500
1294,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,16,1,7.455298,0.000000,2.0,0,1,0,8.371242,9.064274
860,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,9,1,7.757479,5.484797,1.5,1,1,1,9.208639,9.703022


In [15]:
preprocessor.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('MedianImputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('drop_features', ...), ('num_imputer', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the o

In [16]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

lasso = Lasso(alpha=0.0004672260332236319, max_iter=10000)
ridge = Ridge(alpha=9.931760994296951)
lgbm  = lgb.LGBMRegressor(
    n_estimators=1812, learning_rate=0.01185060865103241,
    num_leaves=14, min_child_samples=5,
    subsample=0.9927865856597863, colsample_bytree=0.7009144370133,
    random_state=42, verbose=-1
)

voting_tuned = VotingRegressor([
    ('Lasso',    lasso),
    ('Ridge',    ridge),
    ('LightGBM', lgbm),
])

ml_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model',        voting_tuned),
])

ml_pipeline.fit(X_train, y_train)

y_pred = ml_pipeline.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
print(f"RMSE Voting: {rmse:.4f}")

# Кросс-валидация на всём X
cv_scores = cross_val_score(ml_pipeline, X, y,
                            cv=cv, scoring='neg_root_mean_squared_error')
print(f"RMSE CV Voting: {-cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

RMSE Voting: 0.1250
RMSE CV Voting: 0.1265 ± 0.0274


In [17]:
my_index = []

results = {
    'train': [],
    'valid': []
}

results_cross_val = {
    'cross_val_score': [],
}

results_opt = {
    'optuna_params': [],
    'optuna_score': []
}

# cv_scores = cross_val_score(voting_tuned, X_train, y,cv=cv, scoring='neg_root_mean_squared_error')

# def kv_score(pipeline):    
#     cv = KFold(n_splits=10, random_state=42, shuffle=True)

#     cross_validation_result = cross_val_score(
#         pipeline,
#         X, # Подаем датасет целиком!!! Разделение на train и valid происходит внутри
#         y,
#         cv = cv
#     )

#     return results_cross_val['cross_val_score'].append(cross_validation_result.mean())

lasso = Lasso(alpha=0.0004672260332236319, max_iter=10000)
ridge = Ridge(alpha=9.931760994296951)
lgbm  = lgb.LGBMRegressor(
    n_estimators=1812, learning_rate=0.01185060865103241,
    num_leaves=14, min_child_samples=5,
    subsample=0.9927865856597863, colsample_bytree=0.7009144370133,
    random_state=42, verbose=-1
)

Voting = VotingRegressor([
    ('Lasso',    lasso),
    ('Ridge',    ridge),
    ('LightGBM', lgbm),
])

def fit_and_add(pipeline, x = X_train, y = y_train):
    pipeline.fit(X_train, y_train)
    train_score = accuracy_score(y_train, pipeline.predict(X_train))
    valid_score = accuracy_score(y_valid, pipeline.predict(X_valid))
    results['train'].append(train_score)
    results['valid'].append(valid_score)
    
def my_pipeline(model, name):
    my_index.append(name)
    ml_pipeline = Pipeline(
        [
            ('preprocessor', preprocessor),
            ('model', model)
            ]
        )
    return ml_pipeline

In [ ]:
#fit_and_add(my_pipeline(Voting, 'Voting'))

# results_df = pd.DataFrame(
#     results,
#     index=my_index
# )
# results_df

In [19]:
test = pd.read_csv("../../data/test.csv")

preds_log = ml_pipeline.predict(test)
preds = np.expm1(preds_log)

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": preds
})

submission.to_csv("../../data/submission.csv", index=False)
submission.head(10)

,Id,SalePrice
0,1461,120130.148959
1,1462,156600.444630
2,1463,179536.757877
3,1464,189958.788292
4,1465,198284.624839
5,1466,173481.194303
6,1467,181829.254464
7,1468,162517.104747
8,1469,191560.221849
9,1470,118549.152177


In [20]:
joblib.dump(ml_pipeline, '../../models/voting_pipeline.pkl')
joblib.dump(preprocessor, '../../models/preprocessor.pkl')
print('Pipeline saved!')

Pipeline saved!
